## Student Performance Classification



## 1. Problem Definition



- **Input (features):** student demographic, family, school, study and lifestyle information, plus earlier-period grades (`G1` and `G2`).
- **Target:** `Pass_Fail`
  - `1` = Pass (`G3 >= 10`)
  - `0` = Fail (`G3 < 10`)

### Business question
> Can we use information available about a student to classify whether the student will pass the final course assessment?



In [2]:
# Import the libraries 

import os
import io
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

## 2. Load the Dataset

In [3]:

CSV_NAME = "student-mat.csv"
ZIP_NAME = "student.zip"

def find_csv_in_zip(z):
    # Direct CSV
    csvs = [n for n in z.namelist() if n.lower().endswith(".csv")]
    preferred = [n for n in csvs if "student-mat" in n.lower()]
    if preferred:
        return preferred[0]
    if csvs:
        return csvs[0]

    # Nested ZIP files
    nested = [n for n in z.namelist() if n.lower().endswith(".zip")]
    for nested_name in nested:
        with z.open(nested_name) as f:
            nested_bytes = f.read()
        with zipfile.ZipFile(io.BytesIO(nested_bytes)) as nested_zip:
            found = find_csv_in_zip(nested_zip)
            if found:
                with nested_zip.open(found) as csv_file:
                    return pd.read_csv(csv_file, sep=";")
    return None

if os.path.exists(CSV_NAME):
    df = pd.read_csv(CSV_NAME, sep=";")
elif os.path.exists(ZIP_NAME):
    with zipfile.ZipFile(ZIP_NAME) as z:
        result = find_csv_in_zip(z)
        if isinstance(result, pd.DataFrame):
            df = result
        else:
            with z.open(result) as f:
                df = pd.read_csv(f, sep=";")
else:
    raise FileNotFoundError(
        "Place student-mat.csv or student+performance.zip in the same folder as this notebook."
    )

print("Dataset shape:", df.shape)
display(df.head())

FileNotFoundError: Place student-mat.csv or student+performance.zip in the same folder as this notebook.

## 3. Data Understanding

In [4]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nSummary statistics:")
display(df.describe().T)

NameError: name 'df' is not defined

## 4. Data Cleaning

In [ ]:
# Check for missing values and duplicates

print("Missing values:")
display(df.isnull().sum().sort_values(ascending=False).to_frame("missing_values"))

print("Duplicate rows:", df.duplicated().sum())

# Remove exact duplicate rows
df = df.drop_duplicates().copy()

print("\nShape after removing duplicates:", df.shape)

## 5. Exploratory Data Analysis (EDA)



In [ ]:
# Distribution of final grades

plt.figure(figsize=(8, 5))
plt.hist(df["G3"], bins=range(0, 22), edgecolor="black")
plt.xlabel("Final Grade (G3)")
plt.ylabel("Number of Students")
plt.title("Distribution of Final Grades")
plt.show()

In [ ]:
# Create the classification target

df["Pass_Fail"] = (df["G3"] >= 10).astype(int)

class_counts = df["Pass_Fail"].value_counts().sort_index()
class_labels = ["Fail", "Pass"]

print("Class distribution:")
display(
    class_counts.rename(index={0: "Fail", 1: "Pass"}).to_frame("count")
)

plt.figure(figsize=(6, 4))
plt.bar(class_labels, [class_counts.get(0, 0), class_counts.get(1, 0)])
plt.xlabel("Class")
plt.ylabel("Number of Students")
plt.title("Pass/Fail Class Distribution")
plt.show()

print("Pass rate: {:.2%}".format(df["Pass_Fail"].mean()))

In [ ]:
# Compare earlier grades with the pass/fail target

group_summary = df.groupby("Pass_Fail")[["G1", "G2", "absences", "studytime", "failures"]].mean()
group_summary.index = ["Fail", "Pass"]

display(group_summary)

plt.figure(figsize=(7, 5))
for label, value in [(0, "Fail"), (1, "Pass")]:
    subset = df[df["Pass_Fail"] == label]
    plt.scatter(
        subset["G2"],
        subset["G3"],
        alpha=0.5,
        label=value
    )

plt.xlabel("Second Period Grade (G2)")
plt.ylabel("Final Grade (G3)")
plt.title("G2 vs G3 by Pass/Fail Class")
plt.legend()
plt.show()

## 6. Feature and Target Preparation

The final grade `G3` is used only to create the target and is then removed from the predictors.

We keep `G1` and `G2` because they are earlier-period grades and are available before the final grade. However, they are strongly related to the final result, so this is an important limitation to mention when interpreting model performance.

We do **not** create extra regression targets or regression models because this project is classification-only.

In [ ]:
# Separate features (X) and classification target (y)

X = df.drop(columns=["G3", "Pass_Fail"])
y = df["Pass_Fail"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("\nTarget values:")
print(y.value_counts().rename({0: "Fail", 1: "Pass"}))

## 7. Train-Test Split

In [ ]:
# Use a stratified split so the Pass/Fail proportions are preserved.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining class proportions:")
display(y_train.value_counts(normalize=True).rename({0: "Fail", 1: "Pass"}))

print("\nTesting class proportions:")
display(y_test.value_counts(normalize=True).rename({0: "Fail", 1: "Pass"}))

## 8. Data Preprocessing

Machine learning algorithms require numerical input.

### Numerical features
- Missing values are handled using the median.
- Features are standardized using `StandardScaler`.

### Categorical features
- Missing values are handled using the most frequent category.
- Categories are converted into numerical variables using one-hot encoding.

A `Pipeline` and `ColumnTransformer` are used so that preprocessing is learned **only from the training data**, helping prevent data leakage.

In [ ]:
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Number of categorical features:", len(categorical_features))
print("Number of numerical features:", len(numeric_features))

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeric_features
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
            ]),
            categorical_features
        )
    ]
)

## 9. Baseline Model

A baseline gives us a simple reference point.

The `DummyClassifier` predicts the majority class. A useful ML model should perform better than this simple baseline.

In [ ]:
baseline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DummyClassifier(strategy="most_frequent"))
])

baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

print("Baseline Accuracy:", round(accuracy_score(y_test, baseline_pred), 4))
print("Baseline F1:", round(f1_score(y_test, baseline_pred), 4))

## 10. Classification Models

We compare several standard supervised-learning classification algorithms:

1. **Logistic Regression** – a strong and interpretable linear classification baseline.
2. **K-Nearest Neighbors (KNN)** – classifies observations using nearby training examples.
3. **Decision Tree** – learns rule-based splits and is easy to interpret.
4. **Support Vector Machine (SVM)** – finds a decision boundary that separates classes.
5. **Gaussian Naive Bayes** – a probabilistic classifier based on Bayes' theorem.

All models use the same preprocessing pipeline.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=5000, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=7),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        min_samples_split=5,
        random_state=42
    ),
    "SVM": SVC(
        kernel="rbf",
        C=1.0,
        probability=True,
        random_state=42
    ),
    "Naive Bayes": GaussianNB()
}

results = []
fitted_models = {}

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    probabilities = pipeline.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1": f1_score(y_test, predictions, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, probabilities)
    })

    fitted_models[name] = pipeline

results_df = pd.DataFrame(results).sort_values("F1", ascending=False).reset_index(drop=True)

display(results_df.round(4))

## 11. Cross-Validation

A single train-test split can give a result that depends on the particular observations placed in the test set.

To make the comparison more reliable, we use **5-fold stratified cross-validation** on the training data.

The main comparison metric is **F1-score**, because it balances precision and recall.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = []

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="f1",
        n_jobs=-1
    )

    cv_results.append({
        "Model": name,
        "Mean CV F1": scores.mean(),
        "Std CV F1": scores.std()
    })

cv_results_df = pd.DataFrame(cv_results).sort_values(
    "Mean CV F1", ascending=False
).reset_index(drop=True)

display(cv_results_df.round(4))

## 12. Model Evaluation

The following metrics are used:

- **Accuracy:** proportion of all predictions that are correct.
- **Precision:** among students predicted as Pass, how many actually passed?
- **Recall:** among students who actually passed, how many were correctly identified?
- **F1-score:** harmonic mean of precision and recall.
- **ROC-AUC:** measures how well the model ranks positive cases above negative cases.

For this project, **F1-score is used as the main model-selection metric**, while the other metrics provide additional evidence.

In [ ]:
# Plot model comparison

metrics = ["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]

for metric in metrics:
    plot_df = results_df.sort_values(metric, ascending=False)

    plt.figure(figsize=(9, 5))
    plt.bar(plot_df["Model"], plot_df[metric])
    plt.ylim(0, 1.05)
    plt.ylabel(metric)
    plt.xlabel("Model")
    plt.title(f"Classification Model Comparison - {metric}")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.show()

## 13. Confusion Matrix for the Best Model

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = fitted_models[best_model_name]

best_predictions = best_model.predict(X_test)

print("Best model based on test-set F1:", best_model_name)
print("\nClassification report:")
print(
    classification_report(
        y_test,
        best_predictions,
        target_names=["Fail", "Pass"],
        zero_division=0
    )
)

cm = confusion_matrix(y_test, best_predictions)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Fail", "Pass"]
)

disp.plot(colorbar=False)
plt.title(f"Confusion Matrix - {best_model_name}")
plt.show()

## 14. Feature Importance / Interpretation

For the Decision Tree model, feature importance can provide an intuitive indication of which variables contributed most to the classification.

This section is included as an interpretation step, not as a requirement to use Decision Tree as the final model.

In [ ]:
# Extract and display Decision Tree feature importances

tree_pipeline = fitted_models["Decision Tree"]
tree_model = tree_pipeline.named_steps["model"]
tree_preprocessor = tree_pipeline.named_steps["preprocessor"]

feature_names = tree_preprocessor.get_feature_names_out()

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": tree_model.feature_importances_
}).sort_values("Importance", ascending=False).head(15)

display(importance_df.round(4))

plt.figure(figsize=(9, 6))
plt.barh(
    importance_df["Feature"][::-1],
    importance_df["Importance"][::-1]
)
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top Decision Tree Feature Importances")
plt.tight_layout()
plt.show()

## 15. Final Model Comparison and Conclusion

### How to interpret the results

The model with the highest **F1-score** on the test set is selected as the best-performing classification model for this experiment.

When presenting the project, discuss:

1. Which model achieved the highest F1-score?
2. Did it outperform the baseline?
3. How did precision and recall compare?
4. What does the confusion matrix show?
5. Which features appear important?
6. What limitations should be considered?

### Important limitation

`G1` and `G2` are earlier grades and are strongly related to `G3`. Including them can make classification substantially easier. Therefore, the model should not be interpreted as proving that demographic or lifestyle factors alone determine whether a student passes.

### Final conclusion

This project demonstrates a complete **classification-focused Machine Learning Foundations workflow**: understanding the data, cleaning it, exploring it, defining a binary target, splitting the data, preprocessing numerical and categorical variables, establishing a baseline, training multiple classifiers, using cross-validation, evaluating with classification metrics, and interpreting the final model.

In [ ]:
# Automatically generate a concise final summary

best_row = results_df.iloc[0]
baseline_accuracy = accuracy_score(y_test, baseline_pred)

print("=" * 60)
print("FINAL CLASSIFICATION SUMMARY")
print("=" * 60)
print(f"Best model: {best_row['Model']}")
print(f"Test Accuracy: {best_row['Accuracy']:.4f}")
print(f"Test Precision: {best_row['Precision']:.4f}")
print(f"Test Recall: {best_row['Recall']:.4f}")
print(f"Test F1-score: {best_row['F1']:.4f}")
print(f"Test ROC-AUC: {best_row['ROC_AUC']:.4f}")
print(f"Baseline Accuracy: {baseline_accuracy:.4f}")
print("=" * 60)